In [ ]:
import sys

!"{sys.executable}" -m pip install -q --upgrade pip
!"{sys.executable}" -m pip install -q lime
print("Python:", sys.executable)

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import torch
import torch.nn as nn
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
SEED = 42
N_EXAMPLES = 10
NUM_FEATURES = 10
NUM_SAMPLES = 5000

DATA_DIR = Path("data")
MLP_DIR  = Path("artifacts/mlp")
OUT_DIR  = Path("artifacts/lime")
OUT_DIR.mkdir(parents=True, exist_ok=True)

X_train = np.load(DATA_DIR / "X_train.npy")
y_train = np.load(DATA_DIR / "y_train.npy")
X_test  = np.load(DATA_DIR / "X_test.npy")
y_test  = np.load(DATA_DIR / "y_test.npy")

meta          = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))
feature_names = meta["feature_names"]
class_names   = meta["target_names"]

class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 100),
            nn.ReLU(),
            nn.Linear(100, 50),
            nn.ReLU(),
            nn.Linear(50, 1),
        )

    def forward(self, x):
        return self.net(x)


class MLPWrapper:
    """Sklearn-compatible wrapper that adds predict_proba to the PyTorch MLP."""

    def __init__(self, mlp_model, device):
        self.model = mlp_model
        self.device = device

    def predict_proba(self, X):
        """Return shape (n_samples, 2) probability array."""
        self.model.eval()
        with torch.no_grad():
            t = torch.tensor(np.asarray(X, dtype=np.float32)).to(self.device)
            probs1 = torch.sigmoid(self.model(t)).cpu().numpy().flatten()
        return np.column_stack([1.0 - probs1, probs1])


config = json.loads((MLP_DIR / "model_config.json").read_text(encoding="utf-8"))
device = "cuda" if torch.cuda.is_available() else "cpu"
_mlp = MLP(config["input_dim"]).to(device)
_mlp.load_state_dict(torch.load(MLP_DIR / "model_state.pt", map_location=device))
model = MLPWrapper(_mlp, device)

X_train.shape, X_test.shape, class_names

In [ ]:
rng = np.random.default_rng(SEED)
idx0 = np.where(y_test == 0)[0]
idx1 = np.where(y_test == 1)[0]

half = N_EXAMPLES // 2
n0 = min(len(idx0), half)
n1 = min(len(idx1), N_EXAMPLES - n0)

chosen = []
if n0 > 0:
    chosen.extend(rng.choice(idx0, size=n0, replace=False).tolist())
if n1 > 0:
    chosen.extend(rng.choice(idx1, size=n1, replace=False).tolist())

if len(chosen) < N_EXAMPLES:
    remaining = np.setdiff1d(np.arange(len(y_test)), np.array(chosen, dtype=int))
    extra = rng.choice(remaining, size=(N_EXAMPLES - len(chosen)), replace=False)
    chosen.extend(extra.tolist())

chosen = sorted(set(int(i) for i in chosen))
chosen[:5], len(chosen)

In [ ]:
indices_path = OUT_DIR / "chosen_test_indices.json"
indices_payload = {
    "seed": SEED,
    "n_examples": N_EXAMPLES,
    "chosen_test_indices": chosen,
}
indices_path.write_text(json.dumps(indices_payload, ensure_ascii=False, indent=2), encoding="utf-8")
indices_path.as_posix()

In [ ]:
explainer = LimeTabularExplainer(
    training_data=X_train,
    training_labels=y_train,
    feature_names=feature_names,
    class_names=class_names,
    mode="classification",
    discretize_continuous=True,
    random_state=SEED,
)

def predict_proba_fn(X):
    return model.predict_proba(X)


In [ ]:
results = []
for i in chosen:
    x = X_test[i]
    true_y = int(y_test[i])
    proba = predict_proba_fn(x.reshape(1, -1))[0]
    pred_y = int(np.argmax(proba))

    exp = explainer.explain_instance(
        data_row=x,
        predict_fn=predict_proba_fn,
        num_features=NUM_FEATURES,
        num_samples=NUM_SAMPLES,
        top_labels=1,
    )

    explained_label = int(exp.available_labels()[0])

    weights_str = exp.as_list(label=explained_label)

    local_exp = exp.local_exp[explained_label]
    intercept = exp.intercept[explained_label]

    html_path = OUT_DIR / f"lime_test_{i:03d}.html"
    html_path.write_text(exp.as_html(), encoding="utf-8")

    rec = {
        "test_index": int(i),
        "true_y": true_y,
        "pred_y": pred_y,
        "proba": [float(p) for p in proba],
        "explained_label": explained_label,
        "num_features": int(NUM_FEATURES),
        "num_samples": int(NUM_SAMPLES),
        "weights": [{"feature": f, "weight": float(w)} for f, w in weights_str],
        "local_exp": [{"feature_idx": int(fi), "weight": float(w)} for fi, w in local_exp],
        "intercept": float(intercept),
        "html_file": html_path.name,
    }
    results.append(rec)

summary_path = OUT_DIR / "lime_summary.json"
summary_path.write_text(
    json.dumps(
        {
            "seed": SEED,
            "n_examples": len(results),
            "num_features": NUM_FEATURES,
            "num_samples": NUM_SAMPLES,
            "results": results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

summary_path.as_posix(), len(results)